# 04. Match Prediction Model
This notebook trains a classifier (Logistic Regression) to predict the outcome of a match based on team names, venue, and toss information. The pipeline is saved to disk for deployment in the Streamlit application.


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib

df_matches = pd.read_csv(os.path.join('..', 'data', 'processed', 'cleaned_matches.csv'))
print('Matches loaded.')


## 1. Prepare Target and Features
We filter normal matches and predict whether Team 1 wins (target=1) or Team 2 wins (target=0).


In [ ]:
df_model = df_matches[(df_matches['outcome'] == 'normal') & (df_matches['winner'].notna())].copy()
df_model['target'] = (df_model['winner'] == df_model['team1']).astype(int)

feature_cols = ['team1', 'team2', 'venue', 'toss_winner', 'toss_decision']
X = df_model[feature_cols]
y = df_model['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')


## 2. Train Pipeline and Evaluate
We train a pipeline containing a OneHotEncoder and a Logistic Regression Classifier.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), feature_cols)]
)

match_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

match_pipeline.fit(X_train, y_train)
y_pred = match_pipeline.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.3f}')
print(classification_report(y_test, y_pred))


## 3. Save Model Pipeline
We persist the trained model pipeline.


In [ ]:
joblib.dump(match_pipeline, '../models/match_predictor.pkl')
print('Saved match_predictor.pkl')
